# Product Feature Engineering

## Notebook purpose

This notebook creates product and item-level features for return-risk modelling.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
FEATURES_DIR = DATA_DIR / "features"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Feature output directory: {FEATURES_DIR}")

Project root: C:\Users\Victus\OneDrive\Desktop\E-Commerce-Sales-Analytics
Feature output directory: C:\Users\Victus\OneDrive\Desktop\E-Commerce-Sales-Analytics\data\features


# Load Order-Item And Product Catalog Data

In [2]:
order_items_df = pd.read_csv(DATA_DIR / "order_items.csv")
products_df = pd.read_csv(DATA_DIR / "product_catalog.csv")

print(f"Order-item rows: {len(order_items_df):,}")
print(f"Product catalog rows: {len(products_df):,}")
print(f"Unique orders in items: {order_items_df['order_id'].nunique():,}")
print(f"Unique products in items: {order_items_df['product_id'].nunique():,}")

Order-item rows: 397,569
Product catalog rows: 1,175
Unique orders in items: 138,116
Unique products in items: 1,175


# Validate Product Keys And Join Product Attributes

In [3]:
product_key_summary = pd.Series({
    "product_catalog_rows": len(products_df),
    "unique_product_ids": products_df["product_id"].nunique(),
    "duplicate_product_ids": products_df["product_id"].duplicated().sum(),
    "missing_product_ids_in_items": (
        ~order_items_df["product_id"].isin(products_df["product_id"])
    ).sum()
})

display(product_key_summary)

assert product_key_summary["duplicate_product_ids"] == 0
assert product_key_summary["missing_product_ids_in_items"] == 0

product_catalog_rows            1175
unique_product_ids              1175
duplicate_product_ids              0
missing_product_ids_in_items       0
dtype: int64

In [4]:
item_product_df = order_items_df.merge(
    products_df,
    on="product_id",
    how="left",
    validate="m:1",
    suffixes=("_item", "_catalog"),
    indicator="product_match"
)

print(f"Rows before product join: {len(order_items_df):,}")
print(f"Rows after product join:  {len(item_product_df):,}")

display(item_product_df.head())

Rows before product join: 397,569
Rows after product join:  397,569


,order_id,product_id,quantity,unit_price_item,discount_percentage,discount_amount,gross_sales,tax_amount,shipping_cost,net_sales,product_cost_item,profit,product_name,product_category,product_subcategory,brand,supplier,unit_price_catalog,product_cost_catalog,product_rating,product_match
0,ORD-301242,PROD-000738,2,244.40,0.34,166.00,488.80,22.60,6.48,351.88,173.28,172.12,Garmin Total client-driven task-force Medical ...,Health & Wellness,Medical Devices,Garmin,Euro Logistics,244.40,86.64,4.80,both
1,ORD-301242,PROD-000181,3,287.13,0.36,311.89,861.39,38.47,5.55,593.52,415.05,172.92,Tommy Hilfiger Open-architected full-range foc...,Fashion,Shoes,Tommy Hilfiger,Pacific Trade,287.13,138.35,3.00,both
2,ORD-773460,PROD-000973,4,754.80,0.47,"1,413.89","3,019.20",305.01,9.00,"1,919.32","1,976.16",-65.84,Swarovski Re-engineered discrete monitoring Ea...,Jewelry,Earrings,Swarovski,MultiSource,754.80,494.04,3.50,both
3,ORD-773460,PROD-000268,4,31.36,0.34,42.17,125.44,15.82,0.00,99.09,55.72,43.37,NARS Enterprise-wide actuating firmware Fragra...,Beauty & Personal Care,Fragrances,NARS,Global Trade,31.36,13.93,4.00,both
4,ORD-449374,PROD-001040,3,6.31,0.09,1.70,18.93,1.21,0.98,19.42,5.91,12.53,Mars Multi-tiered homogeneous artificial intel...,Grocery,Pantry Staples,Mars,Asian Manufacturing,6.31,1.97,3.00,both


# Create Product Composition Features

In [5]:
product_features_df = (
    item_product_df
    .groupby("order_id", as_index=False)
    .agg(
        item_count=("product_id", "size"),
        unique_product_count=("product_id", "nunique"),
        total_quantity=("quantity", "sum"),

        category_count=("product_category", "nunique"),
        subcategory_count=("product_subcategory", "nunique"),
        brand_count=("brand", "nunique"),
        supplier_count=("supplier", "nunique"),

        total_gross_sales=("gross_sales", "sum"),
        total_discount_amount=("discount_amount", "sum"),
        total_tax_amount=("tax_amount", "sum"),
        total_shipping_cost=("shipping_cost", "sum"),
        total_item_net_sales=("net_sales", "sum"),
        total_item_product_cost=("product_cost_item", "sum"),
        total_item_profit=("profit", "sum"),

        minimum_item_unit_price=("unit_price_item", "min"),
        maximum_item_unit_price=("unit_price_item", "max"),
        average_item_unit_price=("unit_price_item", "mean"),

        minimum_product_rating=("product_rating", "min"),
        maximum_product_rating=("product_rating", "max"),
        average_product_rating=("product_rating", "mean"),

        maximum_discount_percentage=("discount_percentage", "max"),
        average_discount_percentage=("discount_percentage", "mean")
    )
)

product_features_df["average_quantity_per_item"] = (
    product_features_df["total_quantity"]
    / product_features_df["item_count"]
)

product_features_df["order_discount_rate"] = np.where(
    product_features_df["total_gross_sales"] > 0,
    product_features_df["total_discount_amount"]
    / product_features_df["total_gross_sales"],
    0
)

product_features_df["order_profit_margin_percentage"] = np.where(
    product_features_df["total_item_net_sales"] != 0,
    product_features_df["total_item_profit"]
    / product_features_df["total_item_net_sales"] * 100,
    0
)

display(product_features_df.head())

,order_id,item_count,unique_product_count,total_quantity,category_count,subcategory_count,brand_count,supplier_count,total_gross_sales,total_discount_amount,total_tax_amount,total_shipping_cost,total_item_net_sales,total_item_product_cost,total_item_profit,minimum_item_unit_price,maximum_item_unit_price,average_item_unit_price,minimum_product_rating,maximum_product_rating,average_product_rating,maximum_discount_percentage,average_discount_percentage,average_quantity_per_item,order_discount_rate,order_profit_margin_percentage
0,ORD-100002,2,2,5,2,2,2,2,"1,576.81",0.00,78.84,37.03,"1,692.68",822.86,832.79,233.33,643.49,438.41,4.10,4.70,4.40,0.00,0.00,2.50,0.00,49.20
1,ORD-100004,3,3,3,3,3,3,3,464.88,26.82,78.85,16.46,533.37,267.98,248.93,12.91,335.86,154.96,4.10,4.90,4.47,0.10,0.08,1.00,0.06,46.67
2,ORD-100016,2,2,3,2,2,2,2,413.46,69.19,24.09,18.95,387.31,181.72,186.64,61.00,176.23,118.61,3.70,4.00,3.85,0.18,0.14,1.50,0.17,48.19
3,ORD-100022,3,3,6,3,3,3,3,"1,500.75",396.33,143.58,73.61,"1,321.61",710.63,537.37,155.85,416.72,257.44,2.50,4.10,3.30,0.45,0.26,2.00,0.26,40.66
4,ORD-100023,5,5,11,5,5,5,4,"3,177.67","1,009.44",151.78,27.63,"2,347.64","1,691.48",628.53,13.04,853.43,325.21,3.50,4.80,4.14,0.40,0.24,2.20,0.32,26.77


# Add Dominant Product Category

In [6]:
category_sales_df = (
    item_product_df
    .groupby(
        ["order_id", "product_category"],
        as_index=False
    )
    .agg(
        category_net_sales=("net_sales", "sum")
    )
)

dominant_category_df = (
    category_sales_df
    .sort_values(
        ["order_id", "category_net_sales", "product_category"],
        ascending=[True, False, True]
    )
    .drop_duplicates(subset="order_id")
    .rename(
        columns={
            "product_category": "dominant_product_category"
        }
    )
    [["order_id", "dominant_product_category"]]
)

product_features_df = product_features_df.merge(
    dominant_category_df,
    on="order_id",
    how="left",
    validate="1:1"
)

display(
    product_features_df[
        [
            "order_id",
            "item_count",
            "unique_product_count",
            "category_count",
            "total_quantity",
            "order_discount_rate",
            "average_product_rating",
            "dominant_product_category"
        ]
    ].head()
)

,order_id,item_count,unique_product_count,category_count,total_quantity,order_discount_rate,average_product_rating,dominant_product_category
0,ORD-100002,2,2,2,5,0.00,4.40,Health & Wellness
1,ORD-100004,3,3,3,3,0.06,4.47,Baby & Kids
2,ORD-100016,2,2,2,3,0.17,3.85,Home & Kitchen
3,ORD-100022,3,3,3,6,0.26,3.30,Sports & Outdoors
4,ORD-100023,5,5,5,11,0.32,4.14,Home Appliances


# Validate Product Feature Table

In [7]:
validation_summary = pd.Series({
    "total_rows": len(product_features_df),
    "unique_order_ids": product_features_df["order_id"].nunique(),
    "duplicate_order_ids": product_features_df["order_id"].duplicated().sum(),
    "missing_dominant_category": (
        product_features_df["dominant_product_category"].isna().sum()
    ),
    "contains_return_status": "return_status" in product_features_df.columns,
    "contains_return_label": "return_label" in product_features_df.columns
})

display(validation_summary)

assert validation_summary["total_rows"] == validation_summary["unique_order_ids"]
assert validation_summary["duplicate_order_ids"] == 0
assert validation_summary["contains_return_status"] is False
assert validation_summary["contains_return_label"] is False

total_rows                   138116
unique_order_ids             138116
duplicate_order_ids               0
missing_dominant_category         0
contains_return_status        False
contains_return_label         False
dtype: object

# Review Product Feature Distributions

In [8]:
display(
    product_features_df[
        [
            "item_count",
            "unique_product_count",
            "total_quantity",
            "category_count",
            "total_gross_sales",
            "total_discount_amount",
            "total_item_net_sales",
            "total_item_profit",
            "average_product_rating",
            "order_discount_rate"
        ]
    ].describe()
)

print("Dominant product category distribution:")
display(
    product_features_df["dominant_product_category"]
    .value_counts()
    .rename_axis("dominant_product_category")
    .reset_index(name="order_count")
)

,item_count,unique_product_count,total_quantity,category_count,total_gross_sales,total_discount_amount,total_item_net_sales,total_item_profit,average_product_rating,order_discount_rate
count,"138,116.00","138,116.00","138,116.00","138,116.00","138,116.00","138,116.00","138,116.00","138,116.00","138,116.00","138,116.00"
mean,2.88,2.88,6.10,2.64,"1,493.99",258.11,"1,393.02",598.76,3.68,0.15
std,1.51,1.51,3.89,1.30,"1,405.44",401.57,"1,275.42",561.95,0.48,0.12
min,1.00,1.00,1.00,1.00,6.31,0.00,6.35,"-1,281.59",2.50,0.00
25%,2.00,2.00,3.00,2.00,497.41,30.36,483.13,205.69,3.37,0.07
50%,3.00,3.00,5.00,2.00,"1,078.31",114.34,"1,027.53",441.53,3.68,0.13
75%,4.00,4.00,8.00,3.00,"2,035.37",306.11,"1,898.48",817.18,4.00,0.21
max,18.00,18.00,48.00,11.00,"16,219.29","6,759.02","16,218.00","7,083.68",4.90,0.60


Dominant product category distribution:


,dominant_product_category,order_count
0,Electronics,19361
1,Home Appliances,15440
2,Jewelry,14604
3,Sports & Outdoors,12474
4,Automotive,12310
5,Home & Kitchen,9031
6,Health & Wellness,8459
7,Baby & Kids,8428
8,Office Supplies,8103
9,Fashion,6787


# Save Product Feature Table

In [9]:
output_path = FEATURES_DIR / "product_features.csv"

product_features_df.to_csv(output_path, index=False)

print(f"Saved product features to: {output_path}")
print(f"Output shape: {product_features_df.shape}")

Saved product features to: C:\Users\Victus\OneDrive\Desktop\E-Commerce-Sales-Analytics\data\features\product_features.csv
Output shape: (138116, 27)
